In [3]:
from river import datasets
from river.datasets import synth
from river import evaluate
from river import metrics
from river.drift import ADWIN
from src.streaming_random_patches import SRPClassifier, SRPClassifierSDDM
from src.river_sddm import RiverSDDM

In [4]:
%load_ext autoreload
%autoreload 2

In [5]:
# 1. Set up a Synthetic Stream with abrupt Concept Drift
# We transition from SEA generator function 0 to function 2 at instance 2000
stream = synth.ConceptDriftStream(
    stream=synth.SEA(seed=42, variant=0),
    drift_stream=synth.SEA(seed=42, variant=2),
    position=2000,
    width=50,
    seed=123
).take(4000) # Run for 4000 instances

In [6]:
import numpy as np

class MultiDimClusteringStream:
    def __init__(self, dims=4):
        self.t = 0
        self.dims = dims
        # Definiujemy centra klastrów w przestrzeni 4D
        self.centers = {
            0: np.ones(dims) * 2,  # Klaster 0: [2, 2, 2, 2]
            1: np.ones(dims) * -2  # Klaster 1: [-2, -2, -2, -2]
        }
        
    def get_sample(self):
        self.t += 1
        # Wybór klastra (Y)
        y = np.random.choice([0, 1])
        center = self.centers[y]
        
        if y == 0:
            # Klaster 0 jest stabilny: cechy X są niezależne
            cov = np.eye(self.dims) * 0.5
        else:
            # Klaster 1 dryfuje:
            if self.t < 500:
                # Przed dryfem: cechy niezależne
                cov = np.eye(self.dims) * 0.5
            else:
                # PO DRYFIE: Silna relacja między X[0] a X[1] 
                # oraz przesunięcie środka (Local Concept Drift)
                cov = np.eye(self.dims) * 0.5
                cov[0, 1] = cov[1, 0] = 0.45  # Pojawia się korelacja
                center = center + 0.5         # Klaster "puchnie" i się przesuwa
        
        x = np.random.multivariate_normal(center, cov)
        return self.t, x, y

# Inicjalizacja
stream = MultiDimClusteringStream(dims=4)

for _ in range(1000):
    t, x_vec, y_true = stream.get_sample()
    
    # Symulacja wykrycia zmiany w klastrze nr 1 po kroku 500
    if t == 550:
        # Przykładowy raport z detektora monitorującego statystyki klastrów
        print(f"ALARM: Local drift in feature-label relation!")
        print(f"  - Time step: {t}")
        print(f"  - Target Cluster (Y): {y_true}")
        print(f"  - Feature dimensions (X): {len(x_vec)}")
        print(f"  - Shift detected in: X[0] vs X[1] correlation")
        print(f"  - Internal Variance change: +15.4%")
        print(f"  - Status: Updating local distance metric for Cluster 1")
        print("-" * 40)

ALARM: Local drift in feature-label relation!
  - Time step: 550
  - Target Cluster (Y): 0
  - Feature dimensions (X): 4
  - Shift detected in: X[0] vs X[1] correlation
  - Internal Variance change: +15.4%
  - Status: Updating local distance metric for Cluster 1
----------------------------------------


In [7]:
def make_sddm():
    return RiverSDDM(
        n_bins=10,
        ref_window_size=400,
        cur_window_size=100,
        threshold=0.6
    )

model = SRPClassifierSDDM(
    n_models=10,
    n_clusters=3,
    seed=42,
    sddm_constructor=make_sddm
)

In [8]:
# 3. Track multiple metrics
metric = metrics.Accuracy() + metrics.MacroF1()

In [ ]:
import numpy as np
from river import metrics, cluster, drift

# 1. Definicja strumienia jako generatora
class MultiDimClusteringStream:
    def __init__(self, dims=4, shift = 1000, max_samples=3000):
        self.dims = dims
        self.shift = shift 
        self.max_samples = max_samples
        self.centers = {
            0: np.ones(dims) * 2,
            1: np.ones(dims) * -2
        }

    def __iter__(self):
        for t in range(1, self.max_samples + 1):
            y = np.random.choice([0, 1])
            center = self.centers[y].copy()
            
            if y == 0:
                cov = np.eye(self.dims) * 0.5
            else:
                # Dryf lokalny po kroku 500
                if t < self.shift:
                    cov = np.eye(self.dims) * 0.5
                else:
                    cov = np.eye(self.dims) * 0.5
                    cov[0, 1] = cov[1, 0] = 0.45 # Zmiana relacji X0-X1
                    center += 0.8                # Przesunięcie klastra
            
            x_array = np.random.multivariate_normal(center, cov)
            # Konwersja na słownik (format wymagany przez River)
            x = {f"feat_{i}": val for i, val in enumerate(x_array)}
            yield x, y

# 2. Inicjalizacja komponentów
stream = MultiDimClusteringStream(max_samples=3000)
model = cluster.KMeans(n_clusters=2) # Przykładowy model klastrowania
metric = metrics.Silhouette()        # Metryka jakości klastrów
drift_detector = drift.ADWIN()       # Detektor dryfu
drifts_detected = 0

# 3. Pętla ewaluacyjna (Twój format)
print(f"{'Instance':<10} | {'Metric Value':<20} | Status")
print("-" * 50)

for i, (x, y) in enumerate(stream):
    # Predict (przypisanie do klastra)
    y_pred = model.predict_one(x)
    
    # Update metric
    if y_pred is not None:
        # W klastrowaniu River, metric.update często przyjmuje x i y_pred
        metric.update(x, y_pred, model.centers)
        
    # Train
    model.learn_one(x)

    # Monitorowanie dryfu (na podstawie błędu lub zmian w danych)
    # Tutaj używamy ADWIN do monitorowania wartości pierwszej cechy
    drift_detector.update(x['feat_0'])
    
    if drift_detector.drift_detected:
        drifts_detected += 1
        print(f"ALARM: Drift detected at instance {i+1}!")
        print(f"  - Total drifts: {drifts_detected}")
        print(f"  - Context: Local change in Cluster 1 distribution")
        print("-" * 40)

    # Print progress every 1000 instances
    if (i + 1) % 1000 == 0:
        print(f"{i+1:<10} | {str(metric):<20} | OK")

Instance   | Metric Value         | Status
--------------------------------------------------
ALARM: Drift detected at instance 800!
  - Total drifts: 1
  - Context: Local change in Cluster 1 distribution
----------------------------------------
1000       | Silhouette           | OK
2000       | Silhouette           | OK
3000       | Silhouette           | OK


In [ ]:
# 4. Manual Evaluation Loop (Better for debugging than progressive_val_score)
drifts_detected = 0

for i, (x, y) in enumerate(stream):
    # Predict
    y_pred = model.predict_one(x)
    
    # Update metric
    if y_pred is not None:
        metric.update(y, y_pred)
        
    # Train
    model.learn_one(x, y)

    # Print progress every 1000 instances
    if (i + 1) % 1000 == 0:
        print(f"Instance {i+1} | {metric}")

Instance 1000 | Accuracy: 94.99%
MacroF1: 93.94%
Instance 2000 | Accuracy: 96.40%
MacroF1: 95.70%
[DRIFT ENSEMBLE] Instance: 2081 | Cluster: 0 | Feature: 0 | Mag: 0.6228
[DRIFT ENSEMBLE] Instance: 2734 | Cluster: 2 | Feature: 1 | Mag: 0.6056
Instance 3000 | Accuracy: 96.17%
MacroF1: 95.27%
[DRIFT ENSEMBLE] Instance: 3609 | Cluster: 1 | Feature: 1 | Mag: 0.6042
Instance 4000 | Accuracy: 96.47%
MacroF1: 95.57%
